In [2]:
import pandas as pd
from tqdm import tqdm
import statsmodels.api as sm

In [ ]:
#import pandas_datareader.data as web

# Fetch momentum factor
#mom_raw = web.DataReader("F-F_Momentum_Factor_daily", "famafrench", start="2010-01-01")[0]

# Save raw to file
#mom_raw.to_csv("data/momentum_daily.csv")

#print(f"Saved {mom_raw.shape[0]} rows to data/momentum_daily.csv")
#print(mom_raw.head())

/tmp/ipykernel_269285/1778545194.py:4: FutureWarning: The argument 'date_parser' is deprecated and will be removed in a future version. Please use 'date_format' instead, or read your data in as 'object' dtype and then call 'to_datetime'.
  mom_raw = web.DataReader("F-F_Momentum_Factor_daily", "famafrench", start="2010-01-01")[0]


Saved 4085 rows to data/momentum_daily.csv
             Mom
Date            
2010-01-04  0.58
2010-01-05  0.62
2010-01-06 -0.05
2010-01-07 -0.87
2010-01-08  0.20


In [6]:
mom = pd.read_csv("data/momentum_daily.csv", parse_dates=["Date"], index_col="Date")

# Convert to decimal and rename
mom = mom / 100
mom = mom.rename(columns={"Mom": "umd"})

print(mom.shape)
print(mom.head())

(4085, 1)
               umd
Date              
2010-01-04  0.0058
2010-01-05  0.0062
2010-01-06 -0.0005
2010-01-07 -0.0087
2010-01-08  0.0020


In [8]:
# reimport the other dataset

ff3_raw = pd.read_csv("data/ff3_daily.csv", parse_dates=["Date"], index_col="Date")

# Convert from percentage to decimal
ff3 = ff3_raw / 100
ff3 = ff3.rename(columns={
    "Mkt-RF": "mkt_rf",
    "SMB":    "smb",
    "HML":    "hml",
    "RF":     "rf_ff"
})


df = pd.read_csv("data/sp500_data_from_2010.csv", parse_dates=["Date"])

stocks = (
    df[df["Ticker"] != "^GSPC"][["Date", "Ticker", "Adj Close"]]
    .sort_values(["Ticker", "Date"])
)
stocks["stock_return"] = stocks.groupby("Ticker")["Adj Close"].pct_change()
stocks = stocks.dropna(subset=["stock_return"])


ff3_data = (
    stocks
    .merge(ff3, left_on="Date", right_index=True, how="inner")
)

# Excess return using French's RF
ff3_data["stock_excess"] = ff3_data["stock_return"] - ff3_data["rf_ff"]



In [9]:
# merge

carhart_data = ff3_data.merge(
    mom[["umd"]],
    left_on="Date",
    right_index=True,
    how="inner"
)

print(f"Carhart dataset shape: {carhart_data.shape}")
print(carhart_data[["Date", "Ticker", "stock_excess", "mkt_rf", "smb", "hml", "umd"]].head())

Carhart dataset shape: (1727531, 10)
           Date Ticker  stock_excess  mkt_rf     smb     hml     umd
424  2010-01-05      A     -0.010863  0.0031 -0.0064  0.0122  0.0062
848  2010-01-06      A     -0.003553  0.0013 -0.0023  0.0055 -0.0005
1272 2010-01-07      A     -0.001296  0.0040  0.0009  0.0096 -0.0087
1696 2010-01-08      A     -0.000325  0.0033  0.0036  0.0002  0.0020
2120 2010-01-11      A      0.000649  0.0013 -0.0013 -0.0026 -0.0048


In [10]:
extremes = pd.read_csv("data/sp500_pct_extremes.csv")
extremes["Date"] = pd.to_datetime(extremes["Date"])

# run the regression

In [ ]:
def run_carhart_loop(df, label, min_obs=5):
    results = []
    for ticker, group in df.groupby("Ticker"):
        if len(group) < min_obs:
            continue
        try:
            X = sm.add_constant(group[["mkt_rf", "smb", "hml", "umd"]])
            y = group["stock_excess"]
            model = sm.OLS(y, X).fit()
            results.append({
                "Ticker":       ticker,
                "alpha":        model.params["const"],
                "beta_mkt":     model.params["mkt_rf"],
                "beta_smb":     model.params["smb"],
                "beta_hml":     model.params["hml"],
                "beta_umd":     model.params["umd"],
                "r2":           model.rsquared,
                "adj_r2":       model.rsquared_adj,
                "p_alpha":      model.pvalues["const"],
                "p_beta_mkt":   model.pvalues["mkt_rf"],
                "p_beta_smb":   model.pvalues["smb"],
                "p_beta_hml":   model.pvalues["hml"],
                "p_beta_umd":   model.pvalues["umd"],
                "n":            len(group)
            })
        except Exception:
            continue
    return pd.DataFrame(results)


carhart_full     = run_carhart_loop(carhart_data, "Carhart — Full Sample")

# Merge extremes with carhart data
carhart_extremes = extremes.merge(
    carhart_data[["Date", "Ticker", "stock_excess", "mkt_rf", "smb", "hml", "umd"]],
    on=["Date", "Ticker"],
    how="inner"
)
carhart_pos = carhart_extremes[carhart_extremes["stock_excess"] > 0].copy()
carhart_neg = carhart_extremes[carhart_extremes["stock_excess"] < 0].copy()

carhart_tail_pos = run_carhart_loop(carhart_pos, "Carhart — Positive Tail")
carhart_tail_neg = run_carhart_loop(carhart_neg, "Carhart — Negative Tail")

print(f"Full sample   : {len(carhart_full)}")
print(f"Positive tail : {len(carhart_tail_pos)}")
print(f"Negative tail : {len(carhart_tail_neg)}")

In [13]:
def summarize_carhart(df, label):
    return {
        "sample":           label,
        "n_stocks":         len(df),
        "median_r2":        df["r2"].median(),
        "median_adj_r2":    df["adj_r2"].median(),
        "median_beta_mkt":  df["beta_mkt"].median(),
        "median_beta_smb":  df["beta_smb"].median(),
        "median_beta_hml":  df["beta_hml"].median(),
        "median_beta_umd":  df["beta_umd"].median(),
        "median_alpha_ann": (df["alpha"] * 252).median(),
        "pct_sig_alpha":    (df["p_alpha"] < 0.05).mean() * 100,
        "pct_sig_umd":      (df["p_beta_umd"] < 0.05).mean() * 100,
    }

comparison = pd.DataFrame([
    summarize_carhart(carhart_full,     "Carhart — Full Sample"),
    summarize_carhart(carhart_tail_pos, "Carhart — Positive Tail"),
    summarize_carhart(carhart_tail_neg, "Carhart — Negative Tail"),
])

display(comparison.T)

,0,1,2
sample,Carhart — Full Sample,Carhart — Positive Tail,Carhart — Negative Tail
n_stocks,423,42,61
median_r2,0.376237,0.692238,0.854992
median_adj_r2,0.375625,0.137812,0.289059
median_beta_mkt,0.939983,0.522633,0.158503
median_beta_smb,0.053771,-0.085258,0.601842
median_beta_hml,0.196626,0.001649,0.34015
median_beta_umd,-0.032806,-0.292279,1.104322
median_alpha_ann,0.035172,47.957742,-41.121436
pct_sig_alpha,6.146572,50.0,62.295082
